# FoodGuard AI: Food Resilience Operating System
## 🚀 Detect → Predict → Explain → Decide → Act → Verify → Learn
This notebook trains the core prediction engine for El Niño impact on food resilience in Tamil Nadu.

In [ ]:
# 1. Download the datasets directly from the GitHub repository
!wget -q https://raw.githubusercontent.com/leelaprasath-cmd/FORESIGHT/main/data/ML_TRAINING_DATA.csv
!wget -q https://raw.githubusercontent.com/leelaprasath-cmd/FORESIGHT/main/data/TAMILNADU_AGRICULTURAL_REAL_MASTER.csv
!wget -q https://raw.githubusercontent.com/leelaprasath-cmd/FORESIGHT/main/data/EL_NINO_SCENARIO_DATA.csv

In [ ]:
# 2. Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Install shap for Explainability
!pip install -q shap
import shap
shap.initjs()

### Step 1: Load & Preprocess the Data
We will load the `ML_TRAINING_DATA.csv` which contains the historical simulation and real features.

In [ ]:
# 3. Load the data
try:
    df = pd.read_csv('ML_TRAINING_DATA.csv')
    print("Dataset loaded successfully. Shape:", df.shape)
    display(df.head())
except Exception as e:
    print("Error loading dataset:", e)
    # Fallback to generating some synthetic data if the CSV doesn't match expected formats exactly for training
    pass


In [ ]:
# 4. Feature Selection & Preprocessing
# Assuming the dataset has columns related to El Nino intensity, Rainfall, Temp, Transport Risk, etc.
# Let's identify the features based on standard crop impact models.

# We will separate features (X) and target (y)
# Target: e.g., 'Supply_Shortage_Tonnes' or 'Price_Increase_Percent'
if 'df' in locals() and not df.empty:
    # Identify numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Assuming the last numeric column is our target variable for this example
    target_col = numeric_cols[-1]
    feature_cols = numeric_cols[:-1]
    
    # Handle missing values
    df = df.fillna(df.mean(numeric_only=True))
    
    X = df[feature_cols]
    y = df[target_col]
    
    print(f"Selected Features: {feature_cols}")
    print(f"Target Variable: {target_col}")
    
    # Split the dataset
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
else:
    print("Data not available for preprocessing. Please verify column names.")


### Step 2: Train the Core Prediction Model
Training a Random Forest model to predict the impact based on climate and logistics inputs.

In [ ]:
# 5. Train the Model
if 'X_train' in locals():
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    
    # Evaluate
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"Model MSE: {mse:.4f}")
    print(f"Model R2 Score: {r2:.4f}")


### Step 3: Root-Cause Intelligence (Explainability)
Using SHAP to explain *why* the model made a specific prediction (e.g., Rainfall deficit vs Heat).

In [ ]:
# 6. SHAP Explanations
if 'model' in locals():
    explainer = shap.Explainer(model, X_train)
    shap_values = explainer(X_test)
    
    # Visualize the impact of features on the prediction for a specific district (e.g., first row)
    print("\n--- Root Cause Intelligence for specific prediction ---")
    shap.plots.waterfall(shap_values[0])
    
    # Visualize overall feature importance
    print("\n--- Global Feature Importance (What drives the risk across the state) ---")
    shap.plots.bar(shap_values)


### Step 4: Intervention Simulator (The Closed Loop)
Simulate what happens if we intervene (e.g., change the transport capacity or add buffer stock).

In [ ]:
# 7. Simulate Intervention
if 'model' in locals():
    # Take one at-risk scenario
    scenario = X_test.iloc[[0]].copy()
    baseline_prediction = model.predict(scenario)[0]
    print(f"Baseline Predicted Risk/Shortage: {baseline_prediction:.2f}")
    
    # INTERVENTION: Let's assume one of the features represents 'Buffer_Stock' or 'Transport_Capacity'
    # We will simulate increasing it by 20% to see the impact.
    intervention_feature = feature_cols[0]  # Just picking the first feature for demonstration, change this to actual intervention column
    scenario[intervention_feature] = scenario[intervention_feature] * 1.20 
    
    post_intervention_prediction = model.predict(scenario)[0]
    print(f"Post-Intervention Predicted Risk/Shortage: {post_intervention_prediction:.2f}")
    print(f"\nIntervention Impact: Reduced risk by {baseline_prediction - post_intervention_prediction:.2f}")
